# Episode 8 — Risk I: DV01

Companion notebook for the video. One swap, three ways to ask "how much does it lose if rates rise a basis point?": bump the market quotes (par DV01), bump the zero rates (zero DV01), or use the annuity (PV01). Then we check how far a single DV01 number can be trusted for bigger moves. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** The quotes in `quotes_illustrative.csv` (as in Episode 6) are made up for teaching. They are **not market prices**. Educational material only, not investment advice.

1. Setup · 2. The trade · 3. Par DV01 · 4. Zero DV01 · 5. PV01 from the annuity · 6. Which curve · 7. Bigger moves · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
tenor = "5Y"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()
cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)

out = {"meta": {
    "episode": 8, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

The curve-building code shared by Episodes 6 to 9:

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

## 2. The trade

Receive fixed at par on a 5-year swap against 6-month BBSW. To bump zero rates later, the swap is priced off curves wrapped in a zero spread that starts at 0.

In [ ]:
quotes = pd.read_csv(quotes_file)
curves, H, quote_handles, helpers = build_curves(quotes)

# A zero-rate spread on each curve: 0 for now, so prices are unchanged.
zspread = {k: ql.SimpleQuote(0.0) for k in ("AONIA", "BBSW6M")}
Z = {k: ql.YieldTermStructureHandle(ql.ZeroSpreadedTermStructure(H[k], ql.QuoteHandle(zspread[k]), ql.Continuous))
     for k in zspread}
par = vanilla_swap(tenor, 0.04, 6, Z["BBSW6M"], Z["AONIA"], notional).fairRate()
swap = vanilla_swap(tenor, par, 6, Z["BBSW6M"], Z["AONIA"], notional)
annuity = abs(swap.fixedLegBPS()) / (notional * 1e-4)
out["trade"] = {"tenor": tenor, "notional": notional, "par_pct": par * 100, "npv": swap.NPV(), "annuity": annuity}
pd.Series(out["trade"])

## 3. Par DV01: bump the market quotes

Shift every AONIA swap quote by ±1bp, keep the basis spreads fixed, let all three curves rebuild, and reprice. DV01 here is the value gained when rates *fall* one basis point:

$$\text{DV01} = \frac{V(-1\text{bp}) - V(+1\text{bp})}{2}$$

In [ ]:
aonia_keys = [k for k in quote_handles if k[0] == "AONIA"]

def shift_quotes(bp):
    for k in aonia_keys:
        quote_handles[k].setValue(quote_handles[k].value() + bp / 1e4)

def npv_quotes_shifted(bp):
    shift_quotes(bp)
    v = swap.NPV()
    shift_quotes(-bp)
    return v

par_dv01 = (npv_quotes_shifted(-1) - npv_quotes_shifted(+1)) / 2

## 4. Zero DV01: bump the zero rates

Instead of the quotes, shift the continuously compounded zero rates of both curves the swap uses (AONIA for discounting, 6-month BBSW for forecasting) by ±1bp.

In [ ]:
def npv_zero_shifted(bp, which=("AONIA", "BBSW6M")):
    for k in which:
        zspread[k].setValue(bp / 1e4)
    v = swap.NPV()
    for k in which:
        zspread[k].setValue(0.0)
    return v

zero_dv01 = (npv_zero_shifted(-1) - npv_zero_shifted(+1)) / 2

## 5. PV01 from the annuity

For a swap at par, a 1bp change in the par rate is worth the annuity times the notional times one basis point.

In [ ]:
pv01 = notional * annuity * 1e-4
dv = {"par_dv01": par_dv01, "zero_dv01": zero_dv01, "pv01": pv01,
      "zero_over_par": zero_dv01 / par_dv01, "par_over_pv01": par_dv01 / pv01,
      "par_dv01_per_million": par_dv01 / (notional / 1e6)}
out["dv01"] = dv
pd.Series(dv)

## 6. Which curve carries it?

Shift the zero rates of one curve at a time. For a swap at par, almost all the rate risk is in the forecasting curve: moving only the discount curve changes the value of both legs almost equally.

In [ ]:
by_curve = {"forecast_only": (npv_zero_shifted(-1, ["BBSW6M"]) - npv_zero_shifted(+1, ["BBSW6M"])) / 2,
            "discount_only": (npv_zero_shifted(-1, ["AONIA"]) - npv_zero_shifted(+1, ["AONIA"])) / 2,
            "both": zero_dv01}
out["by_curve"] = by_curve
out["by_curve_rows"] = [{"shift": "6-month BBSW zero rates only", "dv01": by_curve["forecast_only"]},
                        {"shift": "AONIA zero rates only", "dv01": by_curve["discount_only"]},
                        {"shift": "Both curves", "dv01": by_curve["both"]}]
pd.Series(by_curve)

## 7. Bigger moves

Shift the AONIA quotes in steps of 10bp from −100bp to +100bp and fully revalue each time. Compare with the straight line a single DV01 predicts. The gap is convexity.

In [ ]:
shifts = list(range(-100, 101, 10))
full = [npv_quotes_shifted(s) for s in shifts]
linear = [-par_dv01 * s for s in shifts]
conv = {"shifts_bp": shifts, "full_m": [v / 1e6 for v in full], "linear_m": [v / 1e6 for v in linear],
        "gap_k": [(f - l) / 1e3 for f, l in zip(full, linear)], "zero_k": [0.0 for _ in shifts],
        "gap_up100": full[-1] - linear[-1], "gap_down100": full[0] - linear[0],
        "full_up100": full[-1], "full_down100": full[0], "linear_up100": linear[-1], "linear_down100": linear[0],
        "ticks": [{"x": s, "label": ("+" if s > 0 else "") + f"{s}bp"} for s in (-100, -50, 0, 50, 100)]}
out["convexity"] = conv
plt.figure(figsize=(8, 3.5))
plt.plot(shifts, conv["full_m"], label="full revaluation")
plt.plot(shifts, conv["linear_m"], "--", label="DV01 × shift")
plt.xlabel("parallel shift of AONIA quotes (bp)"); plt.ylabel("value, AUD m"); plt.grid(alpha=.3); plt.legend()
pd.Series({k: v for k, v in conv.items() if k.startswith(("gap", "full_up", "full_down"))})

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")